# Practical 13 – RNNs and LSTMs

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def load_course_csv(filename):
    candidates=[Path("data")/filename,Path("Lecture13_RNN_and_LSTM")/"data"/filename]
    local_path=next((p for p in candidates if p.exists()),None)
    url=f"https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture13_RNN_and_LSTM/data/{filename}"
    source=local_path if local_path else url
    print("Loaded from:",source)
    return pd.read_csv(source)

train_df=load_course_csv("Diesel_Price_Train.csv")
test_df=load_course_csv("Diesel_Price_Test.csv")
train_df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers

series=train_df["Delhi Diesel Price"].astype(float).values.reshape(-1,1)
scaler=MinMaxScaler()
scaled=scaler.fit_transform(series).ravel()

WINDOW=30
X_seq,y_seq=[],[]
for i in range(WINDOW,len(scaled)):
    X_seq.append(scaled[i-WINDOW:i])
    y_seq.append(scaled[i])

X_seq=np.array(X_seq)[...,np.newaxis]
y_seq=np.array(y_seq)

model=keras.Sequential([
    layers.Input(shape=(WINDOW,1)),
    layers.LSTM(32),
    layers.Dense(1)
])
model.compile(optimizer="adam",loss="mse")
model.fit(X_seq,y_seq,epochs=10,batch_size=32,validation_split=0.2,verbose=0)

In [ ]:
test_series=test_df["Delhi Diesel Price"].astype(float).values.reshape(-1,1)
combined=np.concatenate([series[-WINDOW:],test_series])
combined_scaled=scaler.transform(combined).ravel()

X_test_seq=[]
for i in range(WINDOW,len(combined_scaled)):
    X_test_seq.append(combined_scaled[i-WINDOW:i])

X_test_seq=np.array(X_test_seq)[...,np.newaxis]
pred_scaled=model.predict(X_test_seq,verbose=0)
pred=scaler.inverse_transform(pred_scaled).ravel()

result=pd.DataFrame({"actual":test_series.ravel(),"predicted":pred})
result.head()

## Exercise
Change the sequence window from 30 to 14 or 60 and compare prediction behavior.